# Module 07 — GPU Bottlenecks and the Roofline Model

This notebook measures memory-oriented vector addition, FP16 GEMM, and projection shapes on a Lightning NVIDIA T4. The goal is to form a bottleneck hypothesis, not to chase one peak number.

## Lab protocol

Warm up kernels, keep tensors on the GPU, use CUDA events, report medians, and record shape, dtype, work, and memory state with every result.

In [ ]:
from pathlib import Path
import sys
module_dir = Path.cwd() / 'code' / 'module-07'
if not (module_dir / 'roofline_benchmarks.py').exists():
    module_dir = Path.cwd()
sys.path.insert(0, str(module_dir))

import torch
from roofline_benchmarks import (
    gemm_experiment, print_roofline_summary, projection_experiment,
    require_cuda, vector_add_experiment,
)

device = require_cuda()
print('Device:', device)
print_roofline_summary()

## 1. Memory-oriented vector addition

FP32 vector add performs one FLOP for 12 useful bytes, so its arithmetic intensity is approximately 0.083 FLOPs/byte.

In [ ]:
vector_add_experiment(device, elements=16_000_000, warmup=15, repeats=50)

## 2. Compute-oriented FP16 GEMM

In [ ]:
gemm_experiment(device, size=4096, warmup=15, repeats=50)

## 3. Projection shape: GEMV-like to GEMM-like

Keep the weight matrix fixed and vary the number of input rows. Compare batch-one decode-like work with larger batched shapes.

In [ ]:
results = projection_experiment(
    device, batches=[1, 4, 16, 32, 64, 128, 256, 512],
    hidden_size=4096, warmup=15, repeats=50,
)

In [ ]:
import matplotlib.pyplot as plt

batches = [int(result.label.rsplit(' ', 1)[-1]) for result in results]
tflops = [result.tflops for result in results]
plt.figure(figsize=(8, 4))
plt.plot(batches, tflops, marker='o')
plt.xlabel('Input rows / token positions')
plt.ylabel('Achieved TFLOP/s')
plt.title('T4 projection efficiency versus batch size')
plt.grid(alpha=0.3)
plt.show()

## Interpretation

Identify one memory-oriented result, one compute-oriented result, and one shape- or launch-sensitive result. These are hypotheses to validate with profiling and an end-to-end serving workload.